# Database Management Systems: Week 11 - In-Depth Notes

## Week 11 Overview: Backup and Recovery

Week 11 focuses on **Backup and Recovery**, a critical aspect of database management that ensures data durability and system availability in the face of failures. Building on the transaction concepts from Week 10, we explore:

1. **Backup Fundamentals** (Module 51): Why backups are needed, types of backup strategies (full, incremental, differential), cold vs hot backup, and the role of transaction logs.
2. **Recovery Fundamentals** (Module 52): Failure classification, storage hierarchy (volatile, non-volatile, stable), and log-based recovery for serial transactions, including checkpoints.
3. **Concurrent Transaction Recovery** (Module 53): Hot backup with transaction logs, recovery for concurrent transactions using redo/undo phases.
4. **Advanced Recovery and Planning** (Module 54): Recovery with early lock release, logical undo logging, and planning parameters for backup/recovery.
5. **RAID** (Module 55): Redundant Array of Independent Disks for high capacity, speed, and reliability; RAID levels and selection.

These concepts are essential for database administrators to design systems that can survive failures and restore data to a consistent state.

---

## Module 51: Backup and Recovery – Part 1: Backup Fundamentals

### 51.1. Why Backup?

Backup is the process of creating a **representative copy of the data** that contains all necessary contents of the database—data tables, control files, log files, etc.—so that unexpected database failures can be handled by recovering from those backups.

**Two primary types of backup:**

1. **Physical Backup:** Copies the raw data files, control files, log files, archives, etc. It's a direct copy of the storage medium.

2. **Logical Backup:** Extracts logical data (tables, procedures, views, etc.) and stores them in a portable format (e.g., export). This is useful for migration and selective restoration.

**Recovery** is the process of restoring the database to its latest known consistent state after a system failure. **Database logs** record all transactions in sequence, enabling recovery.

### 51.2. Reasons for Backup

1. **Disaster Recovery:** To restore data after a catastrophic failure (fire, flood, earthquake).
2. **Business Process Changes:** Developers may need a previous version of the database to understand changes or roll back application modifications.
3. **Auditing:** Organizations need historical data for legal compliance, fraud investigation, etc.
4. **Minimize Downtime:** Without backups, downtime after a failure would be much longer.

### 51.3. What Data to Backup

1. **Business Data:** The primary information—clients, employees, inventories, sales, course details, etc.
2. **System Data:** Environment configuration, system catalog, log files, software dependencies, disk images.
3. **Media Data:** Photographs, videos, sound files, graphics, BLOB data. These are often large.

### 51.4. Backup Strategies

#### 51.4.1. Full Backup

A **full backup** is a complete copy of the entire database—all tables, procedures, functions, system views, etc. It can completely restore the database to the state at the time of backup.

**Characteristics:**
- Must be done at least once before any other backup type.
- Frequency depends on application: daily, weekly, monthly.
- Requires significant downtime because the system is often taken offline or restricted.
- Large storage media requirement.

**Advantages:**
- Complete restore from a single backup.
- No dependencies on other backups.
- Easy to set up and maintain.

**Disadvantages:**
- Long backup time.
- Long downtime.
- Large storage media.

**Example:** A personal laptop with 600 GB of data takes 3-4 hours for a weekly full backup.

#### 51.4.2. Incremental Backup

An **incremental backup** only backs up data that has changed since the **last backup** (of any type). It uses timestamps to identify changed files.

**Example:** After a full backup on Friday, Saturday's incremental backs up changes since Friday, Sunday's incremental backs up changes since Saturday, etc.

**Advantages:**
- Much less storage (maybe 5% of full).
- Much shorter backup time (minutes instead of hours).
- Can be placed at non-peak times.

**Disadvantages:**
- Recovery requires full backup plus all intermediate incremental backups.
- If any incremental backup is lost, recovery is impossible.
- More complex recovery process.

#### 51.4.3. Differential Backup

A **differential backup** backs up all changes that have occurred since the **last full backup**, regardless of intervening incremental backups.

**Example:** If full backup on Friday, then incremental Sat/Sun/Mon, a differential on Tuesday would include changes from Friday to Tuesday (all changes since full).

**Advantages:**
- Fewer backup sets needed for recovery (full + last differential + intervening incremental).
- Faster recovery than pure incremental.

**Disadvantages:**
- More storage than incremental.
- Backup time may approach full backup if done after many days.

### 51.5. Combined Backup Schedules

A typical monthly schedule might be:
- Full backup on first Sunday.
- Incremental backups daily (Mon-Sat).
- Differential backup on subsequent Sundays.
- Incremental daily until next Sunday.
- Repeat.
- Full backup again at month end.

**Recovery requirement:** Full backup + last differential + up to 6 incremental backups.

### 51.6. Cold vs Hot Backup

**Cold Backup:** Backup performed when the database is offline or mostly offline. This is straightforward but requires downtime.

**Hot Backup:** Backup performed while the database is running and serving users. Essential for 24/7 systems like banking, stock trading, real-time systems.

**Hot backup mechanisms:**
- Use **transaction logs** to record all changes.
- Transaction log backup is lighter than full database backup.
- On failure, restore from last cold backup, then replay transaction log to bring database to failure point.

**Advantages of hot backup:**
- Database remains available.
- Point-in-time recovery possible.
- Efficient for dynamic data.

**Disadvantages:**
- May not be feasible for huge, monolithic data.
- Lower fault tolerance during backup.
- Higher setup and maintenance cost.

### 51.7. Transaction Logs

A **transaction log** records every action that changes the database (write, commit, abort) as a sequence of log records. It's much smaller than the database and can be hot-backed up efficiently.

**Key idea:** Cold backup for data; hot backup for transaction logs. On failure: recover from cold backup, then replay transaction log.

This will be detailed in subsequent modules.

---

## Module 52: Recovery – Part 1: Failure Classification and Log-Based Recovery (Serial Transactions)

### 52.1. Failure Classification

Failures can be:

1. **Transaction Failure:**
   - **Logical errors:** Application program errors (e.g., deduct 50 from A but credit 40 to B).
   - **System errors:** Deadlock, resource limits, etc.

2. **System Crash:**
   - Power failure, OS crash, hardware fault.
   - **Fail-stop assumption:** Non-volatile storage contents are assumed not to be corrupted by system crash. Disks are assumed to not fail due to crash.

3. **Disk Failure:**
   - Head crash, media damage.
   - Destruction is assumed detectable via checksums.

### 52.2. Recovery Goal

When a failure occurs, we must ensure **Atomicity, Consistency, Durability**.

**Problem example:** Transfer 50 from A to B.
- Failure may occur after debiting A but before crediting B → database inconsistent.
- Failure just after transaction commits but before data is physically written → lost updates.

**Recovery strategy:**
- **During normal processing:** Record enough information (logs) to recover.
- **After failure:** Use recorded information to restore consistency.

### 52.3. Storage Types

1. **Volatile Storage:** Main memory, cache. Loses data on system crash.
2. **Non-Volatile Storage:** Disk, tape, flash. Survives system crash, but may fail (less frequently).
3. **Stable Storage:** A mythical storage that survives all failures. Approximated by maintaining **multiple copies** on distinct non-volatile media.

**Stable storage implementation:**
- Maintain multiple copies (usually 2) on separate disks.
- If one copy fails, use the other.
- **Write protocol:**
  1. Write to first physical block.
  2. If successful, write to second physical block.
  3. Output considered complete only after second write succeeds.

**Recovery from inconsistent copies:**
- If either copy has bad checksum, overwrite it with the good copy.
- If both correct but different, overwrite second with first.

**To reduce overhead, record in-progress disk writes on non-volatile storage (like NVRAM).**

### 52.4. Data Access Protocol

Transactions access data through buffers:

```
Physical Blocks (disk)  ←→  System Buffer (memory)  ←→  Transaction Local Buffer (private)
```

- **Input(B):** Read block B from disk to system buffer.
- **Output(B):** Write system buffer block B to disk.
- **Read(X):** Transaction reads data item X from system buffer to its local buffer (only first time).
- **Write(X):** Transaction writes local value of X back to system buffer.

**Key:** Output can happen before or after commit; it's independent of transaction commit.

### 52.5. Log-Based Recovery (Immediate Modification)

**Log records:**
- `<Ti start>`: Transaction Ti begins.
- `<Ti, X, V1, V2>`: Ti modifies X from old value V1 to new value V2. Written **before** the actual write to system buffer.
- `<Ti commit>`: Ti commits. The commit record is written to stable storage; only then is Ti considered committed.

**Immediate Modification Scheme:** Updates can be written to the disk **before** commit. Log records are written to stable storage **before** the corresponding data update.

**Transaction Commit:** When commit log record is output to stable storage. All prior log records must already be output.

### 52.6. Recovery Actions

**Undo(Ti):** Restore all data items updated by Ti to their old values, going backwards from the last log record. Used for rollback or incomplete transactions.

**Redo(Ti):** Set data items to new values, going forward. Used to reapply committed transactions after failure.

**During normal rollback:**
- Scan log backwards from end.
- For each update record, write old value to X, write compensation log record `<Ti, X, V1>`.
- Stop when `<Ti start>` found, then write `<Ti abort>`.

**During failure recovery:**
- If log contains `<Ti start>` but no `<Ti commit/abort>` → **undo Ti**.
- If log contains both start and commit/abort → **redo Ti**.

### 52.7. Example of Log-Based Recovery

Consider transaction T0: start, write A (1000 → 950), write B (2000 → 2050), commit.
And T1: start, write C (700 → 600), commit.

Log sequence:
```
<T0 start>
<T0, A, 1000, 950>
<T0, B, 2000, 2050>
<T0 commit>
<T1 start>
<T1, C, 700, 600>
<T1 commit>
```

Data writes may occur out of order; some before commit, some after. But log ensures recoverability.

### 52.8. Checkpoints

To avoid scanning the entire log during recovery, we periodically create **checkpoints**:

1. Stop all update operations.
2. Output all log records in main memory to stable storage.
3. Output all modified buffer blocks to disk.
4. Write a `<checkpoint L>` log record, where L is the list of active transactions.

**Recovery with checkpoints:**
- Transactions that committed before checkpoint can be ignored.
- For transactions active at checkpoint or started after, redo if committed, undo if incomplete.

**Trade-off:** Frequent checkpoints reduce recovery time but increase overhead.

---

## Module 53: Recovery – Part 2: Concurrent Transactions and Hot Backup

### 53.1. Hot Backup with Transaction Logs

Hot backup is performed while the database is running. The key is to back up **transaction logs** (which are small) instead of the entire database.

**Example:**
- Backup starts at state with data from 4321 to 4378.
- While backup in progress, a write request comes to modify location 4325.
- Protocol: First write the change to the transaction log (hot backup), then write to database.
- If crash occurs before writing to database, the log still has the change; after recovery, replay log to apply the missing change.

**Recovery vs Restore:**
- **Recover:** Retrieve backup from backup media (cold backup) and transaction logs.
- **Restore:** Apply the transaction log to the recovered database to bring it to a consistent state.

**Process:**
1. Recover database from last cold backup (may be inconsistent if backup was interrupted).
2. Recover transaction log (hot backup, complete).
3. Replay the transaction log (restore) to apply all changes that occurred after the backup point.
4. Database is now consistent and can be opened.

### 53.2. Recovery for Concurrent Transactions

**Assumption:** If a transaction Ti has modified an item, no other transaction can modify the same item until Ti commits or aborts. This ensures rollback is possible.

**Data access protocol for concurrent transactions:**
- Each transaction has its own private buffer.
- Reads from system buffer into private buffer; writes from private buffer to system buffer.
- System buffer block may contain data modified by multiple transactions.

### 53.3. Logging for Concurrent Transactions

Same as serial, but interleaved log records from multiple transactions. The log is a sequence of records from all transactions.

**Normal rollback:**
- Scan log backwards from end.
- For update record, undo and write compensation log record (CLR).
- When `<Ti start>` found, write `<Ti abort>`.

### 53.4. Checkpoint and Recovery for Concurrent Transactions

**At checkpoint:**
- Stop all updates.
- Output all log records and buffer blocks.
- Write `<checkpoint L>` where L is the list of active transactions.

**At recovery (failure):**
- Transactions committed before checkpoint: ignore.
- Transactions active at checkpoint or started after: need recovery.

**Redo Phase:**
- Start from last checkpoint.
- Redo all transactions that appear in the log (whether committed, aborted, or incomplete) because their buffer changes may have been lost.
- Maintain undo list: initially L, add on start, remove on commit/abort.

**Undo Phase:**
- For transactions in undo list (incomplete), scan log backwards, undo updates, write CLR.
- When `<Ti start>` found, write `<Ti abort>`.

**Why redo incomplete transactions?** Because their buffer modifications were lost, and to undo them we need to first bring the database to the failure point, then undo the incomplete ones.

### 53.5. Example

Log:
```
<T0 start>
<T0, B, 2000, 2050>
<checkpoint {T0, T1}>
<T1, C, 700, 600>
<T1 commit>
<T2 start>
<T2, A, 1000, 950>
<T0, B, 2050, 2000>   (T0 rollback CLR)
<T0 abort>
  --- crash ---
```

At recovery:
- Checkpoint at {T0, T1}. After checkpoint, T1 commits, T2 starts, T0 rolls back.
- Redo phase: redo all entries after checkpoint (including T0's rollback).
- Undo phase: undo list = {T2} (since T2 started but no commit/abort). Undo T2's update.

---

## Module 54: Recovery – Part 3: Early Lock Release and Logical Undo

### 54.1. Early Lock Release

In B+ tree concurrency control, low-level locks are often released **early** (before transaction commit) to increase concurrency. This is not two-phase locking.

**Problem:** If a transaction T1 inserts an entry into a B+ tree node, and then another transaction T2 inserts into the same node before T1 commits, the position of T1's entry may change. Physical undo (restoring the node to its old state) is impossible because it would also undo T2's insert.

**Solution:** Use **logical undo** instead of physical undo for these operations.

- **Physical undo:** Replace the entire node with its old value.
- **Logical undo:** Execute a compensating operation (e.g., insert undone by delete).

### 54.2. Operation Logging

To support logical undo, we log **operations** with unique identifiers.

**Log records for an operation O1:**
```
<Ti, O1, operation-begin>
... (physical update log records)
<Ti, O1, operation-end, U>
```
where **U** is the logical undo operation.

**Example: Insert (K5, RID7) into index I9:**
```
<T1, O1, operation-begin>
<T1, X, 10, K5>
<T1, Y, 45, RID7>
<T1, O1, operation-end, delete from I9 where key=K5 and rid=RID7>
```

### 54.3. Recovery with Logical Undo

**If crash/rollback before operation completes:** operation-end record not found → use physical undo information.

**If crash/rollback after operation completes:** operation-end found → perform logical undo using U, skip physical undo for that operation.

**Redo phase:** Redo all operations physically (using new values).

### 54.4. Transaction Rollback with Logical Undo

When rolling back:
- Scan log backwards.
- For update records: undo, write CLR.
- If encounter operation-end with U: perform logical undo operation U, log the undo as a normal operation, skip previous records until operation-begin.
- If encounter operation-abort: skip until operation-begin.
- When reaching `<Ti start>`, write `<Ti abort>`.

### 54.5. Full Example

T0 starts, update B, operation O1 begin, update C from 700 to 600, operation end with U = (C + 100), T0 aborts before O1 ends? Actually example shows T0 aborts after O1 completes. Then T1 starts, operation O2 updates C from 600 to 400, end with U = (C + 200). T0 aborts, logical undo O1 adds 100 to C → C becomes 500. Then T1 commits, C remains 500 (original 700 - 200 = 500). Effect of T0's decrement is annulled.

### 54.6. Planning for Backup and Recovery

Factors to consider:
- **Importance of data:** More critical data needs more frequent backup.
- **Frequency of change:** Transactions change daily; personal data rarely changes.
- **Speed of recovery required:** Acceptable downtime.
- **Equipment:** Hardware, software resources.
- **Staff:** Skilled employees for backup/restore.
- **Storage location:** On-site vs off-site, multiple copies.

---

## Module 55: RAID – Redundant Array of Independent Disks

### 55.1. Introduction

RAID is a disk organization that manages a large number of disks while providing a view of a **single logical disk**.

**Motivation:**
- **High capacity:** Multiple disks combined.
- **High speed:** Parallel access across disks.
- **High reliability:** Data redundancy (mirroring, parity) so that if one disk fails, data can be recovered.

**Key observation:** The chance that *some* disk out of many fails is higher than a specific disk, but RAID allows recovery from such failure.

**Example:** 100 disks, MTTF=100,000 hours → system MTTF ≈ 1000 hours (41 days). Redundancy is crucial.

### 55.2. Basic Techniques

#### 55.2.1. Mirroring (Shadowing)

- Keep duplicate copies of data on separate disks.
- Logical disk = two physical disks.
- Write must update both; read can come from either (faster).
- If one disk fails, use the other.
- Data loss only if both fail before repair.

**Advantages:** High reliability, fast reads.
**Disadvantages:** 50% space utilization, expensive.

#### 55.2.2. Striping

- Split data across multiple disks at bit, byte, or block level.
- Improves parallelism and throughput.

**Bit-level striping:** Split bits of each byte across 8 disks. High parallelism but poor seek times.
**Byte-level striping:** Split bytes sequentially across disks.
**Block-level striping:** Common; block i goes to disk (i mod n) + 1. Requests for different blocks can run in parallel.

**Advantages:** High throughput.
**Disadvantages:** No redundancy (by itself).

#### 55.2.3. Parity

- Store extra parity information to allow error detection and correction.
- **Odd parity:** Add a bit so total number of 1s is odd.
- If one bit flips, parity detects error.
- Since we know which disk failed, we can XOR the remaining bits (including parity) to reconstruct the lost bit.
- Can be bit-level or block-level parity.

### 55.3. RAID Levels

#### RAID 0: Striping (No Redundancy)

- Data striped across disks.
- 100% capacity utilization.
- Best performance, no fault tolerance.
- Use when data safety not critical (temporary data, high-performance workstations).

#### RAID 1: Mirroring

- Full duplication of data on two disks.
- 50% capacity utilization.
- Excellent fault tolerance (can survive one disk failure).
- High read performance; write performance similar to single disk.
- Use for operating systems, transaction databases.

#### RAID 2: Bit-Level Striping with Hamming Code

- Stripe at bit level, use Hamming code for error correction.
- Hamming code can correct single-bit errors and detect double-bit errors.
- Requires additional parity disks (e.g., 4 data disks + 3 parity disks).
- Not commonly used; superseded by RAID 3/5.

#### RAID 3: Byte-Level Striping with Dedicated Parity

- Byte striping with a single dedicated parity disk.
- Parity computed for each stripe across data disks.
- Cannot service multiple requests simultaneously (single block spread across all disks).
- Not widely used.

#### RAID 4: Block-Level Striping with Dedicated Parity

- Block striping with a single dedicated parity disk.
- Parity block for each row of data blocks.
- Write bottleneck: all writes must update parity disk.
- Not widely used.

#### RAID 5: Block-Level Striping with Distributed Parity

- Block striping, but parity is distributed across all disks.
- Parity for different stripes stored on different disks.
- Better read parallelism (all disks involved in reads).
- Can tolerate one disk failure.
- Write performance still lower due to parity calculation.
- Widely used for data warehousing, web servers.

#### RAID 6: Block-Level Striping with Dual Distributed Parity

- Uses two parity blocks (P and Q) per stripe, using Reed-Solomon codes.
- Distributed across disks.
- Can tolerate two disk failures.
- More complex parity calculation.
- Use for large archives, high availability solutions.

### 55.4. Hybrid RAID

Combine multiple RAID levels (nesting).

- **RAID 01 (0+1):** Mirror of stripes. Lower layer RAID 0, upper layer RAID 1. Better reliability than RAID 0.
- **RAID 10 (1+0):** Stripe of mirrors. Lower layer RAID 1, upper layer RAID 0. High throughput and latency, good fault tolerance. Popular for fast databases.
- **RAID 50 (5+0):** Stripe of RAID 5 arrays.

### 55.5. Choosing a RAID Level

Consider:
- **Fault tolerance:** RAID 0 none; RAID 1,5,10 one disk; RAID 6 two disks.
- **Performance:** RAID 0 best; RAID 1 good reads; RAID 5 good reads, slower writes; RAID 10 high all.
- **Capacity utilization:** RAID 0 100%; RAID 1 50%; RAID 5 (n-1)/n; RAID 6 (n-2)/n.
- **Cost:** RAID 0 cheapest; RAID 1,10 expensive; RAID 5 moderate.

**Common choices:**
- **RAID 1:** Operating systems, transaction databases.
- **RAID 5:** Data warehousing, web servers, archiving.
- **RAID 6:** Large archives, backup to disk.
- **RAID 10:** Fast databases, file servers, application servers.

### 55.6. Important Caveats

- RAID does not provide 100% uptime.
- RAID is not a replacement for backups; it does not protect against data corruption, human error, or security issues.
- RAID does not allow dynamic expansion without rebuilding.
- RAID is not a substitute for virtualization or high-availability failover.

---

## Conclusion

Week 11 has provided a comprehensive understanding of **Backup and Recovery**:

- **Backup strategies** (full, incremental, differential) balance storage, time, and recovery complexity.
- **Hot backup with transaction logs** enables point-in-time recovery for 24/7 systems.
- **Log-based recovery** (undo/redo) ensures atomicity and durability after failures.
- **Checkpoints** reduce recovery time.
- **Logical undo** handles early lock release in concurrent systems.
- **RAID** provides high capacity, speed, and reliability through mirroring, striping, and parity.

This knowledge is essential for designing robust, fault-tolerant database systems that can survive failures and restore data to a consistent state. In the next weeks, we will continue exploring other advanced topics in database management.